# 🎬 AI Movie Trailer Generator v2.0
### Modern | Cinematic | Video-Powered
**Powered by:** Gemini AI · Edge TTS · MoviePy

---
### 📋 What This Notebook Does:
1. Connects to Google Gemini AI
2. Takes your movie details as input
3. Generates a cinematic movie story
4. Creates a trailer script with 3 scenes
5. Extracts narration and converts to voice-over
6. Accepts your uploaded video clips
7. Applies cinematic effects and assembles the trailer
8. Exports a final `.mp4` movie trailer

## 📦 Step 1: Install Dependencies

In [ ]:
# Install all required packages
!pip install -q google-genai
!pip install -q edge-tts
!pip install -q moviepy
!pip install -q opencv-python-headless
!pip install -q rich
print('All packages installed successfully!')

## 📚 Step 2: Import Libraries

In [ ]:
import os
import asyncio
import textwrap
from pathlib import Path

from google import genai
from rich.console import Console
from rich.panel import Panel
from rich.progress import Progress, SpinnerColumn, TextColumn
from rich.table import Table
from rich import print as rprint
import edge_tts
import cv2
import numpy as np

from moviepy.editor import (
    VideoFileClip,
    AudioFileClip,
    concatenate_videoclips,
    CompositeVideoClip,
    TextClip,
    ColorClip
)

from IPython.display import Audio, Video, display, clear_output

print('All libraries imported successfully!')

## 🎨 Step 3: Setup Console UI & Directories

In [ ]:
# ─────────────────────────────────────────────
# 🎨 CONSOLE SETUP (Rich UI)
# ─────────────────────────────────────────────
console = Console()

def print_banner():
    console.print(Panel.fit(
        '[bold cyan]🎬 AI MOVIE TRAILER GENERATOR v2.0[/bold cyan]\n'
        '[dim]Powered by Gemini AI · Edge TTS · MoviePy[/dim]',
        border_style='cyan'
    ))

def print_step(step_num, title, description=''):
    console.print(f'\n[bold yellow]━━━ STEP {step_num}: {title} ━━━[/bold yellow]')
    if description:
        console.print(f'[dim]{description}[/dim]')

def print_success(message):
    console.print(f'[bold green]✅ {message}[/bold green]')

def print_info(message):
    console.print(f'[bold blue]ℹ️  {message}[/bold blue]')

def print_error(message):
    console.print(f'[bold red]❌ {message}[/bold red]')

# ─────────────────────────────────────────────
# 📁 DIRECTORY SETUP
# ─────────────────────────────────────────────
OUTPUT_DIR = Path('/content/trailer_output')
UPLOAD_DIR = Path('/content/trailer_videos')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

VOICE_PATH   = OUTPUT_DIR / 'trailer_voice.mp3'
TRAILER_PATH = OUTPUT_DIR / 'cinematic_trailer.mp4'

print_banner()
print_success('Console UI ready!')
print_success(f'Output directory: {OUTPUT_DIR}')
print_success(f'Upload directory: {UPLOAD_DIR}')

## 🔐 Step 4: Connect to Gemini AI
> ⚠️ **Security Note:** Replace `YOUR_API_KEY_HERE` with your actual Gemini API key.
> Get your key from: https://makersuite.google.com/app/apikey

In [ ]:
import os
from getpass import getpass
from google import genai

# Ask for API key securely
if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

GEMINI_MODEL = "gemini-2.5-flash"

print("✅ Gemini connected securely!")

## 🎭 Step 5: Enter Your Movie Details

In [ ]:
print_step(2, 'MOVIE DETAILS INPUT')

console.print('\n[bold cyan]Please provide details for your movie trailer:[/bold cyan]')

movie_title = input('🎬 Movie Title     : ').strip()
genre       = input('🎭 Genre           : ').strip()
main_actor  = input('⭐ Main Character  : ').strip() or 'the protagonist'
setting     = input('🌍 Setting/World   : ').strip() or 'a mysterious world'
tone        = input('🎵 Tone (e.g. dark/epic/romantic): ').strip() or 'epic'

table = Table(title='🎬 Your Movie Details', border_style='cyan')
table.add_column('Field',  style='bold yellow')
table.add_column('Value',  style='bold white')
table.add_row('Title',     movie_title)
table.add_row('Genre',     genre)
table.add_row('Character', main_actor)
table.add_row('Setting',   setting)
table.add_row('Tone',      tone)
console.print(table)

## 📖 Step 6: Generate Movie Story

In [ ]:
print_step(3, 'GENERATING MOVIE STORY', 'Using Gemini AI to craft your story...')

story_prompt = f"""
You are a Hollywood screenwriter. Create a gripping, cinematic movie story.

Movie Title    : {movie_title}
Genre          : {genre}
Main Character : {main_actor}
Setting        : {setting}
Tone           : {tone}

Requirements:
- Make it emotionally engaging and visually rich
- Include a clear beginning, conflict, and hint at resolution
- Suitable for a blockbuster movie trailer
- Maximum 180 words
- Write in present tense for cinematic impact
"""

with console.status('[bold cyan]Generating story...[/bold cyan]', spinner='dots'):
    story_response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=story_prompt
    )
    movie_story = story_response.text

console.print(Panel(
    movie_story,
    title='[bold cyan]📖 Generated Movie Story[/bold cyan]',
    border_style='cyan',
    padding=(1, 2)
))

## 🎥 Step 7: Generate Trailer Script

In [ ]:
print_step(4, 'GENERATING TRAILER SCRIPT', 'Creating cinematic scenes and narration...')

trailer_prompt = f"""
You are a professional movie trailer editor and copywriter.

Based on this movie story, generate a powerful 3-scene movie trailer script.

Movie Story:
{movie_story}

Tone: {tone}

Return your response in EXACTLY this format (keep the labels identical):

SCENE 1:
VISUAL: [Detailed cinematic shot description]
NARRATION: [Powerful voice-over line]
MOOD: [Emotional beat]

SCENE 2:
VISUAL: [Detailed cinematic shot description]
NARRATION: [Powerful voice-over line]
MOOD: [Emotional beat]

SCENE 3:
VISUAL: [Detailed cinematic shot description]
NARRATION: [Powerful voice-over line]
MOOD: [Emotional beat]

BACKGROUND MUSIC: [Music style, instruments, tempo]
TITLE CARD: [Final on-screen text / tagline]
"""

with console.status('[bold cyan]Generating trailer script...[/bold cyan]', spinner='dots'):
    trailer_response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=trailer_prompt
    )
    trailer_script = trailer_response.text

console.print(Panel(
    trailer_script,
    title='[bold cyan]🎬 Trailer Script[/bold cyan]',
    border_style='magenta',
    padding=(1, 2)
))

## 🎙️ Step 8: Extract Narration

In [ ]:
print_step(5, 'EXTRACTING NARRATION', 'Pulling voice-over lines from the script...')

narration_prompt = f"""
From the movie trailer script below, extract ONLY the narration lines.

Rules:
- Do NOT include scene numbers, visual descriptions, or music notes
- Combine the narration into a smooth, flowing voice-over
- Add natural pauses using '...' between scenes
- Make it sound like a professional movie trailer voice-over
- Keep the tone: {tone}

Trailer Script:
{trailer_script}
"""

with console.status('[bold cyan]Extracting narration...[/bold cyan]', spinner='dots'):
    narration_response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=narration_prompt
    )
    narration_text = narration_response.text

console.print(Panel(
    narration_text,
    title='[bold cyan]🎙️ Voice-Over Narration[/bold cyan]',
    border_style='green',
    padding=(1, 2)
))

## 🔊 Step 9: Generate Voice-Over (TTS)

In [ ]:
print_step(6, 'GENERATING VOICE-OVER', 'Converting narration to cinematic speech...')

VOICE_OPTIONS = {
    'epic'     : 'en-US-GuyNeural',
    'dark'     : 'en-US-GuyNeural',
    'romantic' : 'en-US-JennyNeural',
    'action'   : 'en-US-DavisNeural',
    'thriller' : 'en-US-TonyNeural',
    'default'  : 'en-US-GuyNeural'
}

selected_voice = VOICE_OPTIONS.get(tone.lower(), VOICE_OPTIONS['default'])
print_info(f'Selected voice: {selected_voice}')

async def generate_voice(text, voice, output_path):
    communicate = edge_tts.Communicate(
        text=text,
        voice=voice,
        rate='-5%',
        volume='+10%'
    )
    await communicate.save(str(output_path))

with console.status('[bold cyan]Generating voice-over...[/bold cyan]', spinner='dots'):
    await generate_voice(narration_text, selected_voice, VOICE_PATH)

print_success(f'Voice-over saved to: {VOICE_PATH}')
display(Audio(str(VOICE_PATH)))

## 📤 Step 10: Upload Your Scene Videos
> **Instructions:**
> 1. Click the 📁 Files icon in the left sidebar
> 2. Navigate to `/content/trailer_videos/`
> 3. Upload your 3 video clips and name them:
>    - `scene_1.mp4` — Opening scene
>    - `scene_2.mp4` — Rising action / conflict  
>    - `scene_3.mp4` — Climax / dramatic moment
> 4. Supported formats: `.mp4`, `.mov`, `.avi`, `.mkv`
> 5. Run the next cell when all 3 are uploaded ✅

In [ ]:
print_step(7, 'UPLOAD SCENE VIDEOS')

console.print(Panel(
    '[bold yellow]📤 HOW TO UPLOAD YOUR VIDEOS:[/bold yellow]\n\n'
    '1. Click the [bold]📁 Files[/bold] icon in the left sidebar\n'
    '2. Navigate to: [cyan]/content/trailer_videos/[/cyan]\n'
    '3. Upload your 3 video clips and rename them:\n'
    '   • [green]scene_1.mp4[/green] — Opening scene\n'
    '   • [green]scene_2.mp4[/green] — Rising action / conflict\n'
    '   • [green]scene_3.mp4[/green] — Climax / dramatic moment\n'
    '4. Supported formats: [bold].mp4, .mov, .avi, .mkv[/bold]\n'
    '5. Run the next cell when all 3 are uploaded ✅',
    border_style='yellow',
    padding=(1, 2)
))

## 🔍 Step 11: Validate Uploaded Videos

In [ ]:
print_step(8, 'VALIDATING UPLOADED VIDEOS')

SCENE_PATHS = [
    UPLOAD_DIR / 'scene_1.mp4',
    UPLOAD_DIR / 'scene_2.mp4',
    UPLOAD_DIR / 'scene_3.mp4',
]

def validate_videos(scene_paths):
    all_valid = True
    table = Table(title='📋 Video Validation', border_style='cyan')
    table.add_column('Scene',    style='bold yellow')
    table.add_column('File',     style='white')
    table.add_column('Duration', style='cyan')
    table.add_column('Size',     style='dim')
    table.add_column('Status',   style='bold')

    for i, path in enumerate(scene_paths, 1):
        scene_label = f'Scene {i}'
        if path.exists():
            try:
                clip     = VideoFileClip(str(path))
                duration = f'{clip.duration:.1f}s'
                size_mb  = f'{path.stat().st_size / 1024 / 1024:.1f} MB'
                clip.close()
                table.add_row(scene_label, path.name, duration, size_mb,
                              '[green]✅ Ready[/green]')
            except Exception as e:
                table.add_row(scene_label, path.name, '—', '—',
                              f'[red]❌ Error: {e}[/red]')
                all_valid = False
        else:
            table.add_row(scene_label, path.name, '—', '—',
                          '[red]❌ Not Found[/red]')
            all_valid = False

    console.print(table)
    return all_valid

videos_ready = validate_videos(SCENE_PATHS)

if not videos_ready:
    print_error('Some videos are missing or invalid. Please upload them and re-run this cell.')
else:
    print_success('All 3 scene videos are ready!')

## 🎨 Step 12: Video Effects Engine

In [ ]:
print_step(9, 'LOADING VIDEO EFFECTS ENGINE')

def zoom_in_effect(clip, zoom_ratio=0.03):
    def effect(get_frame, t):
        img   = get_frame(t)
        h, w  = img.shape[:2]
        scale = 1 + zoom_ratio * t
        new_w = int(w / scale)
        new_h = int(h / scale)
        x1    = (w - new_w) // 2
        y1    = (h - new_h) // 2
        cropped = img[y1:y1 + new_h, x1:x1 + new_w]
        return cv2.resize(cropped, (w, h))
    return clip.fl(effect)

def zoom_out_effect(clip, zoom_ratio=0.03):
    def effect(get_frame, t):
        img   = get_frame(t)
        h, w  = img.shape[:2]
        scale = 1 + zoom_ratio * (clip.duration - t)
        new_w = int(w / scale)
        new_h = int(h / scale)
        x1    = (w - new_w) // 2
        y1    = (h - new_h) // 2
        cropped = img[y1:y1 + new_h, x1:x1 + new_w]
        return cv2.resize(cropped, (w, h))
    return clip.fl(effect)

def cinematic_bars(clip, bar_height_ratio=0.08):
    w, h       = clip.size
    bar_height = int(h * bar_height_ratio)
    top_bar    = ColorClip(size=(w, bar_height), color=(0, 0, 0))
    bottom_bar = ColorClip(size=(w, bar_height), color=(0, 0, 0))
    top_bar    = top_bar.set_duration(clip.duration).set_position(('center', 'top'))
    bottom_bar = bottom_bar.set_duration(clip.duration).set_position(('center', 'bottom'))
    return CompositeVideoClip([clip, top_bar, bottom_bar])

def color_grade(clip, brightness=1.05, contrast=1.1, saturation=0.9):
    def grade(get_frame, t):
        img  = get_frame(t).astype(np.float32)
        img  = img * brightness
        img  = (img - 128) * contrast + 128
        img  = np.clip(img, 0, 255).astype(np.uint8)
        hsv  = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[:, :, 1] *= saturation
        hsv  = np.clip(hsv, 0, 255).astype(np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return clip.fl(grade)

def add_title_card(clip, title, tagline='', duration=3):
    try:
        title_clip = TextClip(
            title,
            fontsize=70,
            color='white',
            font='Arial-Bold',
            method='caption',
            size=(clip.w - 100, None)
        ).set_duration(duration).set_position('center')
        if tagline:
            tag_clip = TextClip(
                tagline,
                fontsize=35,
                color='#cccccc',
                font='Arial',
                method='caption',
                size=(clip.w - 100, None)
            ).set_duration(duration).set_position(('center', clip.h // 2 + 60))
            return CompositeVideoClip([clip, title_clip, tag_clip])
        return CompositeVideoClip([clip, title_clip])
    except Exception:
        return clip

print_success('Video effects engine loaded!')

## 🎬 Step 13: Build the Cinematic Trailer

In [ ]:
print_step(10, 'BUILDING YOUR CINEMATIC TRAILER', 'Assembling scenes, effects, and audio...')

def build_trailer(scene_paths, audio_path, output_path,
                  target_resolution=(1280, 720), fps=24):
    print_info('Loading audio...')
    audio          = AudioFileClip(str(audio_path))
    total_duration = audio.duration
    scene_duration = total_duration / len(scene_paths)

    print_info(f'Total duration: {total_duration:.1f}s | Per scene: {scene_duration:.1f}s')

    effects = [
        {'zoom': 'in',  'fade_in': 1.0, 'fade_out': 0.8},
        {'zoom': 'out', 'fade_in': 0.8, 'fade_out': 0.8},
        {'zoom': 'in',  'fade_in': 0.8, 'fade_out': 1.5},
    ]

    processed_clips = []

    for i, (path, fx) in enumerate(zip(scene_paths, effects), 1):
        print_info(f'Processing Scene {i}...')
        raw_clip = VideoFileClip(str(path))

        if raw_clip.duration < scene_duration:
            loops    = int(scene_duration / raw_clip.duration) + 1
            raw_clip = concatenate_videoclips([raw_clip] * loops)

        clip = raw_clip.subclip(0, scene_duration)
        clip = clip.resize(target_resolution)
        clip = clip.without_audio()
        clip = color_grade(clip)

        if fx['zoom'] == 'in':
            clip = zoom_in_effect(clip)
        else:
            clip = zoom_out_effect(clip)

        clip = cinematic_bars(clip)
        clip = clip.fadein(fx['fade_in']).fadeout(fx['fade_out'])
        processed_clips.append(clip)
        print_success(f'Scene {i} processed!')

    print_info('Concatenating scenes...')
    final_video = concatenate_videoclips(processed_clips, method='compose')

    print_info('Adding voice-over audio...')
    final_video = final_video.set_audio(audio)

    print_info(f'Exporting to: {output_path}')
    final_video.write_videofile(
        str(output_path),
        fps=fps,
        codec='libx264',
        audio_codec='aac',
        bitrate='5000k',
        preset='medium',
        verbose=False,
        logger=None
    )

    audio.close()
    for c in processed_clips:
        c.close()

    return output_path

if videos_ready:
    try:
        build_trailer(
            scene_paths=SCENE_PATHS,
            audio_path=VOICE_PATH,
            output_path=TRAILER_PATH,
        )
        print_success(f'🎬 Trailer saved to: {TRAILER_PATH}')
    except Exception as e:
        print_error(f'Build failed: {e}')
        raise
else:
    print_error('Cannot build trailer — videos not validated. Please fix and re-run.')

## 🖥️ Step 14: Preview Your Trailer

In [ ]:
print_step(11, 'PREVIEWING YOUR CINEMATIC TRAILER')

console.print(Panel(
    f'[bold cyan]🎬 {movie_title.upper()}[/bold cyan]\n'
    f'[dim]Genre: {genre} | Tone: {tone}[/dim]\n\n'
    f'[green]Your AI-generated movie trailer is ready![/green]',
    border_style='cyan',
    padding=(1, 2)
))

display(Video(str(TRAILER_PATH), embed=True, width=800))

## 📊 Step 15: Final Summary & Download

In [ ]:
print_step(12, 'PROJECT SUMMARY')

summary = Table(title='🎬 Trailer Generation Summary', border_style='cyan')
summary.add_column('Component',  style='bold yellow')
summary.add_column('Details',    style='white')
summary.add_column('Status',     style='bold green')

summary.add_row('Movie Title',    movie_title,                      '✅')
summary.add_row('Genre',          genre,                            '✅')
summary.add_row('Story',          'Generated by Gemini 2.5 Flash', '✅')
summary.add_row('Trailer Script', '3 Scenes + Music + Title Card', '✅')
summary.add_row('Narration',      'Extracted from script',         '✅')
summary.add_row('Voice-Over',     f'Edge TTS — {selected_voice}',  '✅')
summary.add_row('Scene Videos',   '3 user-uploaded clips',         '✅')
summary.add_row('Effects',        'Zoom + Grade + Bars + Fade',    '✅')
summary.add_row('Output',         str(TRAILER_PATH),               '✅')
console.print(summary)

# ── Download the trailer ──────────────────────────────
from google.colab import files
print_info('Downloading your trailer...')
files.download(str(TRAILER_PATH))
print_success('Download started!')

